In [1]:
import os
import base64
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pathlib import Path
import magic
import open_clip
from PIL import Image
import torch
import faiss
from langchain_community.vectorstores import FAISS
from sentence_transformers import SentenceTransformer
import open_clip
from typing import List,Dict
import numpy as np
from word_handler import Chunk, DocMeta, process_docx



# Force Hugging Face to look directly at your D drive directory bypassing the link
os.environ["HF_HOME"] = r"D:\models\huggingface"
os.environ["TORCH_HOME"] = r"D:\models\torch_models"



C:\Users\shahin\AppData\Local\Temp\ipykernel_19788\284588481.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
load_dotenv()

hf_api_key = os.getenv("HUGGIN_FACE_API")



text_index    = None
text_meta: Dict[int, Chunk] = {}


image_index   = None
image_meta: Dict[int, dict] = {}


doc_meta_store:  Dict[str, DocMeta]        = {}
image_to_chunks: Dict[tuple[str, int], List[Chunk]]      = {}
table_to_chunks: Dict[tuple[str, int], List[Chunk]]      = {}
all_chunks:      List[Chunk] = []

In [4]:
client = OpenAI(
    api_key=hf_api_key,
    base_url="https://router.huggingface.co/v1"
)

model_name = "Qwen/Qwen3-4B-Instruct-2507"
vision_model_name = "Qwen/Qwen3-VL-8B-Instruct"

# Convert local image file to base64 string
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")




In [ ]:
##test#######
response = client.chat.completions.create(
    model= model_name,
    messages= [
        {
            "role":"user", "content":"give me a simple code for rag using langchain"
        }
    ]
)

# print(response.choices[0].message.content)
display(Markdown(response.choices[0].message.content))



local_image_path = r"C:\Users\shahin\Desktop\pics\er.jfif"
base64_image = encode_image_to_base64(local_image_path)
image_data_url = f"data:image/jpeg;base64,{base64_image}"

# Request execution block
response = client.chat.completions.create(
    model=vision_model_name,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Describe what you see in this picture in detail."},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_data_url  # Passes the parsed base64 data string variable
                    }
                }
            ]
        }
    ]
)

display(Markdown(response.choices[0].message.content))

In [5]:


file_path =  r"F:\university\az e riz\گزارش.docx"

path = Path(file_path)

suffix, mime = None, None

if path.exists():
    suffix = path.suffix
    mime = magic.from_file(str(path), mime=True)


def route_file(mime, suffix, path):
    if mime == "application/pdf":
        #pdf
        handle_pdf(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
        "application/msword"
    ]:
        #word
        handle_word(path)


    if mime in [
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        "application/vnd.ms-excel"
    ]:
        handle_excel(path)

    if mime in [
        "application/vnd.openxmlformats-officedocument.presentationml.presentation",
        "application/vnd.ms-powerpoint"
    ]:

        handle_pp(path)

    # if mime.startswith("image/"):
    #     return "Image"
    #
    # if mime.startswith("audio/"):
    #     return "Audio"
    #
    # if mime.startswith("video/"):
    #     return "Video"

    return "invalid"







###########Alternative using model###################
# print(mime, suffix)
# system_prompt = """
# Your role is just to analyse the MIME and the suffix passed to you and detect the file type.
# You must identify if it is Excel, Word document, Powerpoint, Audio, Video, Image or PDF.
# Just return one word.
# If what is provided to you is not valid just return the word: invalid.
# """
#
# response = client.chat.completions.create(
#     model="Qwen/Qwen2.5-7B-Instruct",
#     messages=[
#         {"role": "system", "content": system_prompt},
#         {
#             "role": "user",
#             "content": f"MIME: {mime}, suffix: {suffix}"
#         }
#     ]
# )
#
# print(response.choices[0].message.content)

AttributeError: module 'magic' has no attribute 'from_file'

In [18]:
from word_handler import process_docx

def ingest(chunks: List[Chunk], doc_meta: DocMeta,
           img_to_ch: dict, tbl_to_ch: dict):
    doc_meta_store[doc_meta.doc_id] = doc_meta
    all_chunks.extend(chunks)
    for k, v in img_to_ch.items():
        image_to_chunks.setdefault(k, []).extend(v)
    for k, v in tbl_to_ch.items():
        table_to_chunks.setdefault(k, []).extend(v)

def handle_word(path: str):
    chunks, doc_meta, img_to_ch, tbl_to_ch = process_docx(path)
    ingest(chunks, doc_meta, img_to_ch, tbl_to_ch)
    index_chunks(chunks)
    index_images(chunks)

def handle_excel(path):
    pass

def handle_pp(path):
    pass


def handle_pdf(path):
    pass

In [7]:
def handle_chunk(chunk, doc_meta):


    content = chunk.text

    if chunk.image_refs:
        content+= "\n[Image]\n" + str(list(chunk.image_refs.values()))

    if chunk.table_refs:
        content += "\n[TABLES]\n" + str(list(chunk.table_refs.values()))

    return Document




**Embeddings models**


1. mulitlangual e5-v2 for sentences
2. open-clip for images


In [8]:
from sentence_transformers import SentenceTransformer

e5_model = SentenceTransformer("intfloat/multilingual-e5-base")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="laion2b_s34b_b79k"
)

clip_model.eval()


def embed_text(texts: List[str], is_query: bool = False) -> np.ndarray:

    prefix  = "query: " if is_query else "passage: "
    prefixed = [prefix + t for t in texts]
    vecs = e5_model.encode(prefixed, normalize_embeddings=True)
    return np.array(vecs, dtype="float32")

def embed_image(image_paths: List[str]) -> np.ndarray:
    images = torch.stack([
        clip_preprocess(Image.open(p).convert("RGB"))
        for p in image_paths
    ])                                              # shape (n, 3, 224, 224)

    with torch.no_grad():
        vecs = clip_model.encode_image(images)     # shape (n, 512)

    vecs = vecs /np.linalg.norm(vecs, keepdims=True) # L2 normalise
    return vecs.cpu().numpy().astype("float32")


**Faiss Docstores**

In [9]:

def index_chunks(chunks: List[Chunk]):
    global text_index

    texts = [c.text for c in chunks]
    vecs  = embed_text(texts, is_query=False)

    if text_index is None:
        text_index = faiss.IndexFlatIP(vecs.shape[1])

    base = text_index.ntotal
    text_index.add(vecs)
    for i, chunk in enumerate(chunks):
        text_meta[base + i] = chunk

def index_images(chunks: List[Chunk]):
    global image_index

    paths, records = [], []
    for c in chunks:
        if not c.image_refs:
            continue
        for img_id, path in c.image_refs.items():
            if os.path.exists(path):
                paths.append(path)
                records.append({
                    "image_id":    img_id,
                    "path":        path,
                    "doc_id":      c.doc_id,
                    "chunk_index": c.chunk_index,
                    "section_path": c.section_path,
                })

    if not paths:
        return

    vecs = embed_image(paths)

    if image_index is None:
        image_index = faiss.IndexFlatIP(vecs.shape[1])

    base = image_index.ntotal
    image_index.add(vecs)
    for i, rec in enumerate(records):
        image_meta[base + i] = rec

**retreival**

In [19]:
def search_text(query:str, k:int =5) -> List[dict]:
    if text_index == None or text_index.ntotal ==0:
        return []

    q_vec = embed_text(query, is_query=True)

    scores , id = text_index.search(q_vec, k)

    results = []

    for score, fid in zip(scores[0], id[0]):
        if fid == -1:
            continue
        chunk = text_meta[fid]
        results.append({
            "score":        round(float(score), 4),
            "text":         chunk.text,
            "doc_id":       chunk.doc_id,
            "chunk_index":  chunk.chunk_index,
            "chunk_type":   chunk.chunk_type,
            "section_path": chunk.section_path,
            "image_refs":   chunk.image_refs,
            "table_refs":   chunk.table_refs,
        })
    return results

def search_image(query: str, k: int = 5) -> List[dict]:
    if image_index is None or image_index.ntotal == 0:
        return []
    q_token = open_clip.tokenize([query])

    with torch.no_grad:
        q_vec = clip_model.encode_text(q_token)
    q_vec = q_vec/np.linalg.norm(q_vec, keepdims=True)
    q_vec = q_vec.cpu().numpy().astype("float32")

    scores, ids = image_index.search(q_vec, k)

    results = []
    for score, fid in zip(scores[0], ids[0]):
        if fid == -1:
            continue
        meta = image_meta[fid]
        results.append({
            "score":        round(float(score), 4),
            "image_id":     meta["image_id"],
            "path":         meta["path"],
            "doc_id":       meta["doc_id"],
            "chunk_index":  meta["chunk_index"],
            "section_path": meta["section_path"],
        })
    return results

def build_context(text_results:List[dict], image_results:List[dict]) ->str:
    parts = []


    for r in text_results:
        path = " > ".join(r["section_path"]) if r["section_path"] else r["doc_id"]
        parts.append(f"[{path}]\n{r['text']}")

    for r in image_results:
          key = (r["doc_id"], r["image_id"])
          chunks  = image_to_chunks.get(key,[])
          if chunks:
              related = chunks[0]
              path=">".join(r["section_path"] if r["section_path"] else r["doc_id"])
              parts.append(f"[{path}| image_{r["image_id"]}]\n context: {related.text}")

    return "\n___\n".join(parts)




In [20]:

if __name__ == "__main__":
    handle_word(r"F:\university\az e riz\گزارش.docx")

    results = search_text("روش‌شناسی تحقیق", k=3)
    for r in results:
        print(f"score={r['score']} | {r['section_path']}")
        print(f"  {r['text'][:120]}\n")

    context = build_context(results, [])
    print(context[:600])


score=0.7403 | []
  5. حال پس از build و  compile  کردن برنامه به پروتئوس رفته و اجزای زیر را در ابتدا قرار میدهیم.: پردازنده atmega64، یک س

score=0.7384 | []
  بخش دوم: روشن شدن شدن همه نمایشگر ها و شمارش از 2000 تا 0: ایده اصلی این است که در یک لحظه همزمان همه سون سگمت ها روشن ب

score=0.7383 | []
  بخش دوم: روشن شدن شدن همه نمایشگر ها و شمارش از 2000 تا 0: ایده اصلی این است که در یک لحظه همزمان همه سون سگمت ها روشن ب

[گزارش.docx]
5. حال پس از build و  compile  کردن برنامه به پروتئوس رفته و اجزای زیر را در ابتدا قرار میدهیم.: پردازنده atmega64، یک سون سگمت 4 تایی(در آزمایش های بعدی به آن نیاز داریم ، نماینده GND و VDD 6. حال پایه های متناظر در پردازنده و سون سگمت را به هم وصل میکنیم(از پورت c0 تا c6z را به پایه های a تا  g سون سگمت و پورت  های B0 تا B3  را به پایه های 1 تا 4 سون سگمت ) 7. حال فایل .hex را  که از مرحله قبل تولید شده در برنامه قرار داده و نتیجه را مشاهده میکنیم.
[TABLE_22] 
 d|v|b|a
h|g|f|r
l|k|j|i
5|3|2|1
___
[گزارش.docx]
بخش دوم: روشن شدن شدن همه نمایشگر ها و شمارش